# DART 사업보고서 섹션 추출 (정식 OpenDART API 사용)

## 사전 준비
1. **OpenDART API Key 발급** : https://opendart.fss.or.kr → 회원가입 → 인증키 신청 (무료, 즉시 발급)
2. **CSV 위치** : `./data/raw/company_master.csv`
3. **출력 위치** : `./data/processed/dart_reports/{stockcode}_{fiscal_year}.json` (자동 생성)
4. **필요 패키지** : `pip install requests`

## 동작 방식
- `corpCode.xml` API → stock_code → corp_code 매핑 (1회 다운로드, 캐시됨)
- `list.json` API → corp_code + fiscal_year → rcept_no 조회 ([기재정정] 우선, [첨부정정] 회피)
- `document.xml` API → rcept_no → 보고서 ZIP (전체 XML) 다운로드
- ZIP 내 모든 XML을 파싱하여 `<SECTION-1>` 중 TITLE이 II/IV/VI인 것만 추출
- 각 행마다 즉시 JSON 저장 (배치 X), `skip-if-exists` 적용

## Cell 1 — 설정 및 임포트

In [25]:
import os
import re
import io
import csv
import json
import time
import zipfile
import xml.etree.ElementTree as ET
from html import unescape
from pathlib import Path

import requests


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return
    with open(env_path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip("'").strip('"'))


# ===== 사용자가 설정해야 하는 값 =====
def find_project_root() -> Path:
    """노트북 실행 위치가 프로젝트 루트/하위 폴더 어디든 안정적으로 루트 탐색."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').exists() or (candidate / 'data').exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()
load_env_file(PROJECT_ROOT / '.env')
API_KEY = os.getenv('DART_API_KEY', os.getenv('OPENDART_API_KEY', 'PUT_YOUR_API_KEY_HERE'))
# 또는 직접 입력:  API_KEY = 'abcd1234...' (40자)

if API_KEY == 'PUT_YOUR_API_KEY_HERE' or len(API_KEY) != 40:
    raise RuntimeError('OpenDART API Key를 설정해 주세요. 프로젝트 루트의 .env에서 DART_API_KEY 또는 환경변수 OPENDART_API_KEY로 읽을 수 있습니다.')

# ===== 경로 =====
CSV_PATH         = PROJECT_ROOT / 'data/raw/company_master.csv'
CORP_CODE_PATH   = PROJECT_ROOT / 'data/raw/CORPCODE.xml'
OUTPUT_DIR       = PROJECT_ROOT / 'data/processed/dart_reports'
FAIL_LOG_PATH    = OUTPUT_DIR / '_failed_rows.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Rate Limiting =====
SLEEP_BETWEEN_CALLS = 0.3   # OpenDART는 일 20,000회 한도. 0.3초 간격이면 충분히 안전
REQUEST_TIMEOUT     = 60

OPENDART_BASE = 'https://opendart.fss.or.kr/api'

print(f'API Key (앞 8자리): {API_KEY[:8]}...')
print(f'출력 경로: {OUTPUT_DIR.resolve()}')
print(f'CSV 경로: {CSV_PATH.resolve()}')
print(f'CSV 존재: {CSV_PATH.exists()}')

API Key (앞 8자리): 42a1b845...
출력 경로: /Users/home1/Documents/esg-dart-project/data/processed/dart_reports
CSV 경로: /Users/home1/Documents/esg-dart-project/data/raw/company_master.csv
CSV 존재: True


## Cell 2 — corp_code 다운로드 및 매핑

OpenDART의 `corpCode.xml` API는 전체 기업 목록을 ZIP으로 반환합니다 (수MB). 1회만 다운로드하면 됩니다.

In [26]:
def download_corp_code_xml():
    """OpenDART에서 전체 corp_code 목록 다운로드 (ZIP → CORPCODE.xml)."""
    if CORP_CODE_PATH.exists():
        print(f'이미 존재: {CORP_CODE_PATH}')
        return
    print('corpCode.xml 다운로드 중...')
    url = f'{OPENDART_BASE}/corpCode.xml'
    r = requests.get(url, params={'crtfc_key': API_KEY}, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    # 응답이 ZIP 파일임
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extract('CORPCODE.xml', path=CORP_CODE_PATH.parent)
    print(f'저장 완료: {CORP_CODE_PATH}  ({CORP_CODE_PATH.stat().st_size:,} bytes)')


def build_stock_to_corp_map() -> dict:
    """CORPCODE.xml을 파싱하여 {stock_code: corp_code} 딕셔너리 반환.
    동일 stock_code가 여러 corp_code로 매핑될 경우 modify_date가 가장 최근인 것을 채택.
    """
    if not CORP_CODE_PATH.exists():
        download_corp_code_xml()
    tree = ET.parse(CORP_CODE_PATH)
    root = tree.getroot()
    best: dict = {}  # stock_code -> {'corp_code', 'modify_date'}
    for item in root.findall('list'):
        sc = (item.findtext('stock_code') or '').strip()
        cc = (item.findtext('corp_code') or '').strip()
        md = (item.findtext('modify_date') or '').strip()
        if sc and len(sc) == 6 and cc:
            if sc not in best or md > best[sc]['modify_date']:
                best[sc] = {'corp_code': cc, 'modify_date': md}
    return {k: v['corp_code'] for k, v in best.items()}


download_corp_code_xml()
CORP_MAP = build_stock_to_corp_map()
print(f'\n매핑된 stock_code 수: {len(CORP_MAP):,}')
print(f'예시: 005930(삼성전자) -> {CORP_MAP.get("005930")}')

이미 존재: /Users/home1/Documents/esg-dart-project/data/raw/CORPCODE.xml

매핑된 stock_code 수: 3,963
예시: 005930(삼성전자) -> 00126380


## Cell 3 — rcept_no 조회 (`list.json` API)

**보고서 우선순위:**
1. `[기재정정]사업보고서` — 본문 내용까지 정정된 최신본 (가장 정확)
2. `사업보고서` — 원본
3. 그 외 — `[첨부정정]` 등 (본문이 없을 수 있어 마지막 후보)

각 티어 내에서는 가장 최근 접수본을 선택합니다.

In [27]:
def _opendart_get(path: str, params: dict, retries: int = 2) -> dict:
    """OpenDART REST API GET (JSON 응답)."""
    full_params = {'crtfc_key': API_KEY, **params}
    last_err = None
    for attempt in range(retries + 1):
        try:
            r = requests.get(f'{OPENDART_BASE}/{path}', params=full_params, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            last_err = e
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'OpenDART {path} 호출 실패: {last_err}')


def find_rcept_no(corp_code: str, fiscal_year: str) -> str | None:
    """사업보고서 rcept_no 조회. 정정신고 우선순위 적용."""
    bgn = f'{int(fiscal_year) + 1}0101'
    end = f'{int(fiscal_year) + 1}0731'
    data = _opendart_get('list.json', {
        'corp_code': corp_code,
        'bgn_de': bgn,
        'end_de': end,
        'pblntf_detail_ty': 'A001',  # 사업보고서
        'page_count': '100',
    })
    status = data.get('status')
    # 013 = 조회된 데이터 없음, 014 = 정보가 없음
    if status in ('013', '014'):
        # 검색 기간을 넓혀서 한 번 더 시도
        end2 = f'{int(fiscal_year) + 2}0731'
        data = _opendart_get('list.json', {
            'corp_code': corp_code, 'bgn_de': bgn, 'end_de': end2,
            'pblntf_detail_ty': 'A001', 'page_count': '100',
        })
        if data.get('status') in ('013', '014'):
            return None
    if data.get('status') != '000':
        raise RuntimeError(f'list.json 에러: {data.get("status")} {data.get("message")}')

    items = data.get('list', [])
    if not items:
        return None

    def tier(name: str) -> int:
        if '[기재정정]' in name:
            return 1   # 본문까지 정정한 최신본
        if name.strip().startswith('사업보고서') or name.strip().startswith('[첨부추가]사업보고서'):
            return 2   # 원본
        return 3       # [첨부정정] 등 (본문 없을 수 있음)

    # tier 오름차순, 같은 tier 내에서는 rcept_no 내림차순(최신 우선)
    items.sort(key=lambda x: (tier(x.get('report_nm', '')), -int(x.get('rcept_no', '0'))))
    return items[0]['rcept_no']


# 테스트
test_corp = CORP_MAP.get('005930')  # 삼성전자
print(f'삼성전자 2022 사업보고서 rcept_no: {find_rcept_no(test_corp, "2022")}')
print(f'삼성전자 2023 사업보고서 rcept_no: {find_rcept_no(test_corp, "2023")}')

삼성전자 2022 사업보고서 rcept_no: 20230307000542
삼성전자 2023 사업보고서 rcept_no: 20240312000736


## Cell 4 — 보고서 XML 다운로드 및 섹션 추출 (`document.xml` API)

`document.xml`은 ZIP 파일을 반환하며, 그 안에 보고서 본문 XML이 1개 이상 포함되어 있습니다.
각 XML을 파싱하여 `<SECTION-1>`을 순회하고, TITLE이 II/IV/VI에 해당하는 것만 추출합니다.

In [28]:
def fetch_report_zip(rcept_no: str) -> list[tuple[str, str]]:
    """document.xml API 호출. 반환값: [(파일명, XML텍스트), ...]"""
    url = f'{OPENDART_BASE}/document.xml'
    last_err = None
    for attempt in range(3):
        try:
            r = requests.get(url, params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                             timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            content = r.content
            # 정상 응답이면 ZIP, 에러 시에는 status XML이 옴
            if not content[:2] == b'PK':
                # XML 에러 응답
                snippet = content[:500].decode('utf-8', errors='replace')
                raise RuntimeError(f'document.xml 에러 응답: {snippet}')
            files = []
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                for name in z.namelist():
                    if not name.lower().endswith('.xml'):
                        continue
                    raw = z.read(name)
                    text = None
                    for enc in ('utf-8', 'euc-kr', 'cp949'):
                        try:
                            text = raw.decode(enc)
                            break
                        except UnicodeDecodeError:
                            continue
                    if text is None:
                        text = raw.decode('utf-8', errors='replace')
                    files.append((name, text))
            return files
        except (requests.RequestException, zipfile.BadZipFile) as e:
            last_err = e
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'document.xml 다운로드 실패: {last_err}')


def classify_section(title: str) -> str | None:
    """섹션 TITLE이 II/IV/VI 중 어디에 해당하는지 판별. 표기 변형 허용."""
    if not title:
        return None
    t = title.strip()
    # 로마자 변형 정규화 (Ⅱ → II 등)
    t = (t.replace('Ⅱ', 'II').replace('Ⅳ', 'IV').replace('Ⅵ', 'VI')
          .replace('ⅱ', 'II').replace('ⅳ', 'IV').replace('ⅵ', 'VI'))
    tn = re.sub(r'\s+', '', t)  # 공백 제거 후 매칭
    if re.match(r'^II\.?사업의?내용', tn):
        return 'II'
    if re.match(r'^IV\.?이사의?경영진단', tn):
        return 'IV'
    if re.match(r'^VI\.?이사회', tn):  # "이사회 등 회사의 기관 [및 계열회사]에 관한 사항"
        return 'VI'
    return None


def element_to_text(elem) -> str:
    """XML element를 plain text로 변환. 블록 요소 끝에 줄바꿈을 넣음."""
    raw = ET.tostring(elem, encoding='unicode', method='xml')
    # 블록 요소 종료 태그를 줄바꿈으로
    raw = re.sub(r'</(P|TITLE|TR|LI|H[1-6])>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<BR\s*/?>', '\n', raw, flags=re.IGNORECASE)
    # 나머지 태그 제거
    text = re.sub(r'<[^>]+>', ' ', raw)
    text = unescape(text)
    # 공백 정규화
    text = re.sub(r'[ \t\r\f]+', ' ', text)
    text = re.sub(r'\n[ \t]*', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def _local_name(tag: str) -> str:
    """XML namespace가 붙어도 태그명만 비교하기 위한 helper."""
    return tag.rsplit('}', 1)[-1].upper()


def _direct_title_text(elem) -> str:
    """SECTION-1 바로 아래 TITLE만 읽어 섹션 제목을 판별."""
    for child in list(elem):
        if _local_name(child.tag) == 'TITLE':
            return ''.join(child.itertext()).strip()
    return ''


def _normalize_text(text: str) -> str:
    text = unescape(text)
    text = re.sub(r'[ \t\r\f]+', ' ', text)
    text = re.sub(r'\n[ \t]*', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def _extract_sections_with_parser(content: str) -> dict:
    result = {'II': None, 'IV': None, 'VI': None}
    try:
        root = ET.fromstring(content.encode('utf-8'))
    except ET.ParseError:
        return result

    for elem in root.iter():
        if _local_name(elem.tag) != 'SECTION-1':
            continue
        label = classify_section(_direct_title_text(elem))
        if label and result[label] is None:
            result[label] = element_to_text(elem)
    return result


def _extract_sections_with_regex(content: str) -> dict:
    """파서가 실패한 XML을 위한 보조 추출. SECTION-1 시작 위치 기준으로 다음 SECTION-1 전까지 자름."""
    result = {'II': None, 'IV': None, 'VI': None}
    starts = [m.start() for m in re.finditer(r'<SECTION-1\b[^>]*>', content, flags=re.I)]
    starts.append(len(content))
    for i in range(len(starts) - 1):
        chunk = content[starts[i]:starts[i + 1]]
        title_match = re.search(r'<TITLE\b[^>]*>(.*?)</TITLE>', chunk, flags=re.S | re.I)
        if title_match is None:
            continue
        title = unescape(re.sub(r'<[^>]+>', ' ', title_match.group(1)))
        label = classify_section(title)
        if label and result[label] is None:
            text = re.sub(r'</(P|TITLE|TR|LI|H[1-6])>', '\n', chunk, flags=re.I)
            text = re.sub(r'<BR\s*/?>', '\n', text, flags=re.I)
            text = re.sub(r'<[^>]+>', ' ', text)
            result[label] = _normalize_text(text)
    return result


def extract_target_sections(xml_files: list[tuple[str, str]]) -> dict:
    """ZIP 내 모든 XML을 순회하며 SECTION-1 중 TITLE이 II/IV/VI인 것을 추출."""
    result = {'II': None, 'IV': None, 'VI': None}
    for fname, content in xml_files:
        parsed = _extract_sections_with_parser(content)
        fallback = _extract_sections_with_regex(content) if any(v is None for v in parsed.values()) else {}
        for label in result:
            if result[label] is None:
                result[label] = parsed.get(label) or fallback.get(label)
    return result


# 테스트 (삼성전자 2022)
_test_files = fetch_report_zip('20230307000542')
print(f'다운로드된 XML 파일 수: {len(_test_files)}')
for fn, content in _test_files:
    print(f'  {fn}  ({len(content):,} chars)')
_test_sections = extract_target_sections(_test_files)
for k, v in _test_sections.items():
    head = v[:80].replace('\n', ' ⏎ ') if v else '<null>'
    print(f'\n  [{k}] {len(v) if v else 0:,} chars : {head}')

다운로드된 XML 파일 수: 3
  20230307000542.xml  (3,900,326 chars)
  20230307000542_00760.xml  (510,547 chars)
  20230307000542_00761.xml  (593,039 chars)

  [II] 37,087 chars : II. 사업의 내용 ⏎  ⏎ 1. 사업의 개요 ⏎  ⏎ 당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총

  [IV] 18,037 chars : IV. 이사의 경영진단 및 분석의견 ⏎  ⏎ 1. 예측정보에 대한 주의사항 본 자료는 미래에 대한 '예측정보'를 포함하고 있습니다.이는 과거가 아닌 미

  [VI] 32,457 chars : VI. 이사회 등 회사의 기관에 관한 사항 ⏎  ⏎ 1. 이사회에 관한 사항 ⏎  ⏎ 가. 이사회 구성 개요 당사의 이사회는 사내이사 5인(한종희, 경계현, 


## Cell 5 — 메인 배치 처리 함수

- 행 단위로 즉시 JSON 저장
- `skip-if-exists` 적용 (이미 처리된 행은 건너뜀)
- 에러 발생 시 `_failed_rows.csv`에 기록 후 다음 행으로 (재시도 루프 없음)

In [29]:
def append_failure(idx: int, sc: str, fy: str, rcept_no: str | None, reason: str):
    new = not FAIL_LOG_PATH.exists()
    with open(FAIL_LOG_PATH, 'a', encoding='utf-8', newline='') as f:
        w = csv.writer(f)
        if new:
            w.writerow(['idx', 'stock_code', 'fiscal_year', 'rcept_no', 'reason'])
        w.writerow([idx, sc, fy, rcept_no or '', reason])


def process_row(idx: int, row: dict) -> str:
    sc = str(row.get('stock_code', '')).zfill(6)
    fy = str(row.get('fiscal_year', '')).strip()
    company = row.get('company_name', '')
    out_path = OUTPUT_DIR / f'{sc}_{fy}.json'

    if out_path.exists():
        try:
            with open(out_path, 'r', encoding='utf-8') as f:
                existing = json.load(f)
            existing_sections = existing.get('sections', {})
            invalid_existing = [k for k in ('II', 'IV', 'VI') if not existing_sections.get(k) or len(existing_sections.get(k, '')) < 2000]
            if not invalid_existing:
                return 'SKIP (이미 존재)'
            print(f'  기존 파일 재생성 대상: {out_path.name} ({invalid_existing})')
        except Exception as e:
            print(f'  기존 파일 읽기 실패, 재생성: {out_path.name} ({e})')

    cc = CORP_MAP.get(sc)
    if not cc:
        append_failure(idx, sc, fy, None, 'corp_code 미존재')
        return 'FAIL: corp_code 없음'

    # 1) rcept_no 조회
    try:
        rcept_no = find_rcept_no(cc, fy)
    except Exception as e:
        append_failure(idx, sc, fy, None, f'rcept_lookup: {e}')
        return f'FAIL: rcept_lookup {e}'
    if not rcept_no:
        append_failure(idx, sc, fy, None, 'rcept_no 없음')
        return 'FAIL: rcept_no 없음'
    time.sleep(SLEEP_BETWEEN_CALLS)

    # 2) 보고서 ZIP 다운로드 및 섹션 추출
    try:
        xml_files = fetch_report_zip(rcept_no)
        sections = extract_target_sections(xml_files)
        invalid_sections = [k for k, v in sections.items() if not v or len(v) < 2000]
        if invalid_sections:
            reason = f'section_missing_or_short: {invalid_sections}'
            append_failure(idx, sc, fy, rcept_no, reason)
            return f'FAIL: {reason}'
    except Exception as e:
        append_failure(idx, sc, fy, rcept_no, f'fetch: {e}')
        return f'FAIL: fetch {e}'
    time.sleep(SLEEP_BETWEEN_CALLS)

    # 3) JSON 저장 (atomic)
    out = {
        'stock_code': sc,
        'fiscal_year': fy,
        'rcept_no': rcept_no,
        'sections': sections,
    }
    tmp = out_path.with_suffix('.json.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    tmp.replace(out_path)

    sizes = ' '.join(
        f'{k}={len(v):,}' if v else f'{k}=null'
        for k, v in sections.items()
    )
    return f'OK   rcept={rcept_no}  {sizes}'


def process_batch(start_idx: int, end_idx: int):
    """company_master.csv 의 start_idx ~ end_idx 행을 처리 (1-indexed, inclusive)."""
    if not CSV_PATH.exists():
        raise FileNotFoundError(f'CSV 파일이 없습니다: {CSV_PATH.resolve()}')

    with open(CSV_PATH, 'r', encoding='utf-8-sig') as f:
        all_rows = list(csv.DictReader(f))
    target = all_rows[start_idx - 1: end_idx]

    print(f'=== 배치 시작: 행 {start_idx}-{end_idx}  ({len(target)}건) ===\n')
    counts = {'OK': 0, 'SKIP': 0, 'FAIL': 0}
    t0 = time.time()
    for i, row in enumerate(target, 1):
        idx = start_idx + i - 1
        sc = str(row.get('stock_code', '')).zfill(6)
        fy = str(row.get('fiscal_year', '')).strip()
        company = row.get('company_name', '')
        try:
            status = process_row(idx, row)
        except Exception as e:
            status = f'FAIL: 미처리예외 {e}'
            append_failure(idx, sc, fy, None, f'unhandled: {e}')

        if status.startswith('OK'):
            counts['OK'] += 1
        elif status.startswith('SKIP'):
            counts['SKIP'] += 1
        else:
            counts['FAIL'] += 1
        print(f'[{i:3d}/{len(target)}] idx={idx:3d}  {sc} {fy}  {company:<15}  {status}')

    dt = time.time() - t0
    print(f'\n=== 완료: OK={counts["OK"]}, SKIP={counts["SKIP"]}, FAIL={counts["FAIL"]}  ({dt:.0f}초) ===')
    return counts

## Cell 6 — 검증 함수

사용자 요청 검증 항목:
- `"II. 사업의 내용" not in text` → 잘못된 섹션 추출
- `len(text) < 5000` → 비정상적으로 짧음

추가 체크:
- 섹션 헤더 시작 (`II.`, `IV.`, `VI.`)
- 다른 섹션 헤더 (`III.`, `V.`, `VII.`)가 본문 안에 섞이지 않았는지

In [30]:
EXPECTED_HEAD = {
    'II': '사업의 내용',
    'IV': '이사의 경영진단',
    'VI': '이사회',
}

# 다른 섹션의 시작 마커 (본문 중간에 등장하면 잘못된 추출)
OTHER_SECTION_RE = re.compile(
    r'(?:^|\n)\s*(III|V|VII|VIII|IX|X|XI|XII)\.\s+[가-힣]'
)


def validate_outputs(min_len: int = 2000, verbose: bool = True) -> dict:
    files = sorted(p for p in OUTPUT_DIR.glob('*.json') if not p.name.startswith('_'))
    issues = []
    section_lens = {'II': [], 'IV': [], 'VI': []}

    for fp in files:
        with open(fp, 'r', encoding='utf-8') as f:
            d = json.load(f)
        for k in ('II', 'IV', 'VI'):
            v = d['sections'].get(k)
            if v is None:
                issues.append(f'{fp.name}: [{k}] is null')
                continue
            section_lens[k].append(len(v))
            head = v[:200]

            # 1) 헤더 키워드 포함 확인
            if EXPECTED_HEAD[k] not in head:
                issues.append(f'{fp.name}: [{k}] 헤더 누락 ("{EXPECTED_HEAD[k]}" 없음). head={head[:80]!r}')

            # 2) 길이 임계값
            if len(v) < min_len:
                issues.append(f'{fp.name}: [{k}] 너무 짧음 ({len(v)} chars < {min_len})')

            # 3) 다른 섹션 헤더 침범 (앞 100자에는 없어야 함)
            tail_for_check = v[100:]  # 처음 100자(자기 헤더)는 제외
            m = OTHER_SECTION_RE.search(tail_for_check[:5000])  # 앞 5000자 안에 다른 섹션 시작이 보이면 의심
            if m:
                issues.append(f'{fp.name}: [{k}] 다른 섹션 헤더 발견({m.group(1)}.) at offset {100 + m.start()}')

    if verbose:
        print(f'검증 대상: {len(files)}개 JSON 파일')
        print(f'발견된 이슈: {len(issues)}건\n')
        for iss in issues[:50]:
            print(f'  ⚠ {iss}')
        if len(issues) > 50:
            print(f'  ... (이슈 {len(issues) - 50}건 더 있음)')
        print()
        for k, lens in section_lens.items():
            if lens:
                print(f'  [{k}] count={len(lens)}  min={min(lens):,}  max={max(lens):,}  '
                      f'avg={sum(lens) // len(lens):,}  median={sorted(lens)[len(lens) // 2]:,}')
    return {'files': len(files), 'issues': issues, 'section_lens': section_lens}

## Cell 7~14 — 50건씩 순차 실행

**총 381행 → 50건씩 8개 배치로 분할.** 각 셀을 순서대로 실행하세요.
각 배치는 약 3~5분 소요됩니다.

In [31]:
process_batch(1, 50)

=== 배치 시작: 행 1-50  (50건) ===

[  1/50] idx=  1  005930 2022  삼성전자             OK   rcept=20230307000542  II=37,087 IV=18,037 VI=32,457
[  2/50] idx=  2  005930 2023  삼성전자             OK   rcept=20240312000736  II=39,238 IV=17,629 VI=26,804
[  3/50] idx=  3  005930 2024  삼성전자             OK   rcept=20250311001085  II=38,940 IV=16,440 VI=31,543
[  4/50] idx=  4  001460 2022  BYC              OK   rcept=20230316001312  II=7,959 IV=5,146 VI=8,891
[  5/50] idx=  5  001460 2023  BYC              OK   rcept=20240314001341  II=7,744 IV=4,830 VI=9,950
[  6/50] idx=  6  001460 2024  BYC              OK   rcept=20250320001026  II=12,095 IV=4,824 VI=12,291
[  7/50] idx=  7  005380 2022  현대자동차            OK   rcept=20230315001030  II=56,860 IV=12,688 VI=23,771
[  8/50] idx=  8  005380 2023  현대자동차            OK   rcept=20240314001531  II=56,860 IV=12,688 VI=23,771
[  9/50] idx=  9  005380 2024  현대자동차            OK   rcept=20250312001148  II=50,621 IV=15,740 VI=27,791
[ 10/50] idx= 10  000120 2022  C

{'OK': 50, 'SKIP': 0, 'FAIL': 0}

In [32]:
process_batch(51, 100)

=== 배치 시작: 행 51-100  (50건) ===

[  1/50] idx= 51  051910 2024  LG화학             OK   rcept=20250731000339  II=56,687 IV=19,219 VI=19,623
[  2/50] idx= 52  096770 2022  SK이노베이션          OK   rcept=20230322000906  II=93,605 IV=87,372 VI=35,215
[  3/50] idx= 53  096770 2023  SK이노베이션          OK   rcept=20240320000950  II=101,279 IV=116,601 VI=34,682
[  4/50] idx= 54  096770 2024  SK이노베이션          OK   rcept=20250318000862  II=126,197 IV=113,607 VI=38,501
[  5/50] idx= 55  011170 2022  롯데케미칼            OK   rcept=20230321001511  II=55,592 IV=19,815 VI=25,538
[  6/50] idx= 56  011170 2023  롯데케미칼            OK   rcept=20240401004380  II=60,884 IV=21,555 VI=25,084
[  7/50] idx= 57  011170 2024  롯데케미칼            OK   rcept=20250725000806  II=59,867 IV=17,638 VI=25,510
[  8/50] idx= 58  010130 2022  고려아연             OK   rcept=20230309000731  II=54,932 IV=5,482 VI=12,536
[  9/50] idx= 59  010130 2023  고려아연             OK   rcept=20240328001400  II=64,020 IV=6,595 VI=14,241
[ 10/50] idx= 60  010

{'OK': 50, 'SKIP': 0, 'FAIL': 0}

In [33]:
process_batch(101, 150)

=== 배치 시작: 행 101-150  (50건) ===

[  1/50] idx=101  011210 2023  현대위아             OK   rcept=20240329002076  II=33,865 IV=9,656 VI=16,270
[  2/50] idx=102  011210 2024  현대위아             OK   rcept=20250313000158  II=34,631 IV=9,215 VI=15,394
[  3/50] idx=103  014790 2022  HL D&I           OK   rcept=20230315001407  II=25,739 IV=9,572 VI=14,020
[  4/50] idx=104  014790 2023  HL D&I           OK   rcept=20240318000780  II=24,792 IV=9,895 VI=14,410
[  5/50] idx=105  014790 2024  HL D&I           OK   rcept=20250317000836  II=25,770 IV=7,772 VI=14,667
[  6/50] idx=106  030200 2022  케이티              OK   rcept=20230323001449  II=161,569 IV=26,585 VI=25,801
[  7/50] idx=107  030200 2023  케이티              OK   rcept=20240320002050  II=172,078 IV=26,475 VI=28,487
[  8/50] idx=108  030200 2024  케이티              OK   rcept=20250321001834  II=103,617 IV=27,149 VI=22,001
[  9/50] idx=109  042670 2022  현대두산인프라코어        OK   rcept=20230317000838  II=30,226 IV=14,230 VI=16,806
[ 10/50] idx=110  042670

{'OK': 50, 'SKIP': 0, 'FAIL': 0}

In [34]:
process_batch(151, 200)

=== 배치 시작: 행 151-200  (50건) ===

[  1/50] idx=151  001440 2022  대한전선             OK   rcept=20230322001136  II=18,048 IV=8,309 VI=11,773
[  2/50] idx=152  001440 2023  대한전선             OK   rcept=20240321001692  II=19,092 IV=9,129 VI=15,891
[  3/50] idx=153  001440 2024  대한전선             OK   rcept=20250325001056  II=18,954 IV=8,137 VI=14,673
[  4/50] idx=154  001450 2022  현대해상             OK   rcept=20230309000797  II=25,921 IV=33,289 VI=20,443
[  5/50] idx=155  001450 2023  현대해상             OK   rcept=20240314001515  II=22,420 IV=28,087 VI=19,863
[  6/50] idx=156  001450 2024  현대해상             OK   rcept=20250407001731  II=23,323 IV=27,017 VI=20,713
[  7/50] idx=157  001630 2022  종근당홀딩스           OK   rcept=20230314001191  II=121,990 IV=13,125 VI=9,600
[  8/50] idx=158  001630 2023  종근당홀딩스           OK   rcept=20240320001873  II=129,489 IV=13,503 VI=10,400
[  9/50] idx=159  001630 2024  종근당홀딩스           OK   rcept=20250318001235  II=132,581 IV=12,019 VI=11,731
[ 10/50] idx=160  00168

{'OK': 50, 'SKIP': 0, 'FAIL': 0}

In [35]:
process_batch(201, 250)

=== 배치 시작: 행 201-250  (50건) ===

[  1/50] idx=201  000700 2024  유수홀딩스            OK   rcept=20250321001882  II=24,232 IV=14,472 VI=10,234
[  2/50] idx=202  000810 2022  삼성화재해상보험         OK   rcept=20230309000772  II=45,479 IV=35,201 VI=21,525
[  3/50] idx=203  000810 2023  삼성화재해상보험         OK   rcept=20240429000782  II=46,353 IV=38,695 VI=23,302
[  4/50] idx=204  000810 2024  삼성화재해상보험         OK   rcept=20250328001868  II=45,521 IV=36,547 VI=23,493
[  5/50] idx=205  000860 2022  강남제비스코           OK   rcept=20230320000691  II=45,228 IV=15,093 VI=11,043
[  6/50] idx=206  000860 2023  강남제비스코           OK   rcept=20240320001810  II=46,144 IV=15,697 VI=11,019
[  7/50] idx=207  000860 2024  강남제비스코           OK   rcept=20250318001036  II=45,578 IV=15,880 VI=11,809
[  8/50] idx=208  000880 2022  한화               OK   rcept=20230321001313  II=275,890 IV=45,825 VI=23,280
[  9/50] idx=209  000880 2023  한화               OK   rcept=20240430000953  II=283,184 IV=51,155 VI=23,480
[ 10/50] idx=210  00

{'OK': 50, 'SKIP': 0, 'FAIL': 0}

In [36]:
process_batch(251, 300)

=== 배치 시작: 행 251-300  (50건) ===

[  1/50] idx=251  004370 2023  농심               OK   rcept=20240314000698  II=36,410 IV=8,868 VI=18,269
[  2/50] idx=252  004370 2024  농심               OK   rcept=20250312000711  II=35,941 IV=10,011 VI=19,361
[  3/50] idx=253  004540 2022  깨끗한나라            OK   rcept=20230316000673  II=15,877 IV=9,664 VI=8,995
[  4/50] idx=254  004540 2023  깨끗한나라            OK   rcept=20240401001062  II=16,649 IV=8,703 VI=8,553
[  5/50] idx=255  004540 2024  깨끗한나라            OK   rcept=20250320000849  II=21,105 IV=8,546 VI=8,378
[  6/50] idx=256  005090 2022  SGC에너지           OK   rcept=20230309000859  II=38,350 IV=17,791 VI=11,791
[  7/50] idx=257  005090 2023  SGC에너지           OK   rcept=20240313001769  II=28,716 IV=19,915 VI=11,661
[  8/50] idx=258  005090 2024  SGC에너지           OK   rcept=20250312001090  II=27,553 IV=27,672 VI=10,933
[  9/50] idx=259  005290 2022  동진쎄미켐            OK   rcept=20230316001000  II=23,087 IV=9,368 VI=15,448
[ 10/50] idx=260  005290 2023 

{'OK': 50, 'SKIP': 0, 'FAIL': 0}

In [37]:
process_batch(301, 350)

=== 배치 시작: 행 301-350  (50건) ===

[  1/50] idx=301  001020 2022  페이퍼코리아           OK   rcept=20230323001248  II=19,221 IV=6,868 VI=8,362
[  2/50] idx=302  001020 2023  페이퍼코리아           OK   rcept=20240321001475  II=18,426 IV=6,858 VI=8,773
[  3/50] idx=303  001020 2024  페이퍼코리아           OK   rcept=20250320001403  II=16,968 IV=5,961 VI=7,429
[  4/50] idx=304  001070 2022  대한방직             OK   rcept=20230316001199  II=14,193 IV=5,575 VI=10,420
[  5/50] idx=305  001070 2023  대한방직             OK   rcept=20240314001520  II=12,487 IV=6,270 VI=8,727
[  6/50] idx=306  001070 2024  대한방직             OK   rcept=20250313001407  II=12,579 IV=5,275 VI=6,813
[  7/50] idx=307  001130 2022  대한제분             OK   rcept=20230316001412  II=21,551 IV=7,665 VI=5,219
[  8/50] idx=308  001130 2023  대한제분             OK   rcept=20240510001061  II=21,533 IV=7,931 VI=5,144
[  9/50] idx=309  001130 2024  대한제분             OK   rcept=20250313001408  II=20,686 IV=7,791 VI=8,530
[ 10/50] idx=310  001200 2022  유진증권    

{'OK': 47, 'SKIP': 0, 'FAIL': 3}

In [38]:
process_batch(351, 381)

=== 배치 시작: 행 351-381  (31건) ===

[  1/31] idx=351  000970 2024  한국주철관공업          OK   rcept=20250320000304  II=17,487 IV=4,200 VI=5,861
[  2/31] idx=352  001080 2022  만호제강             OK   rcept=20231005000147  II=9,130 IV=3,778 VI=4,682
[  3/31] idx=353  001080 2023  만호제강             OK   rcept=20240920000456  II=10,599 IV=3,578 VI=5,526
[  4/31] idx=354  001080 2024  만호제강             OK   rcept=20250930000631  II=10,577 IV=3,564 VI=11,802
[  5/31] idx=355  001140 2022  국보               OK   rcept=20230321001053  II=16,025 IV=8,896 VI=20,490
[  6/31] idx=356  001140 2023  국보               OK   rcept=20240321001955  II=11,665 IV=8,603 VI=14,501
[  7/31] idx=357  001140 2024  국보               OK   rcept=20250321001993  II=6,862 IV=6,642 VI=13,513
[  8/31] idx=358  001210 2022  금호전기             OK   rcept=20230321001127  II=20,504 IV=7,417 VI=10,135
[  9/31] idx=359  001210 2023  금호전기             OK   rcept=20240320001009  II=21,218 IV=7,568 VI=10,784
[ 10/31] idx=360  001210 2024  금호전기 

{'OK': 31, 'SKIP': 0, 'FAIL': 0}

## Cell 15 — 전체 검증

모든 배치 완료 후 실행하여 잘못된 추출이 있는지 확인.

In [39]:
report = validate_outputs(min_len=2000)

검증 대상: 378개 JSON 파일
발견된 이슈: 39건

  ⚠ 000180_2024.json: [IV] 다른 섹션 헤더 발견(III.) at offset 4098
  ⚠ 000210_2022.json: [IV] 다른 섹션 헤더 발견(III.) at offset 3991
  ⚠ 000210_2023.json: [IV] 다른 섹션 헤더 발견(III.) at offset 3757
  ⚠ 000210_2024.json: [IV] 다른 섹션 헤더 발견(III.) at offset 3753
  ⚠ 000400_2024.json: [IV] 다른 섹션 헤더 발견(X.) at offset 2014
  ⚠ 000590_2022.json: [IV] 다른 섹션 헤더 발견(XI.) at offset 1590
  ⚠ 000590_2023.json: [IV] 다른 섹션 헤더 발견(XI.) at offset 1557
  ⚠ 000590_2024.json: [IV] 다른 섹션 헤더 발견(XI.) at offset 1533
  ⚠ 000650_2022.json: [IV] 다른 섹션 헤더 발견(III.) at offset 2242
  ⚠ 000650_2023.json: [IV] 다른 섹션 헤더 발견(III.) at offset 2212
  ⚠ 000650_2024.json: [IV] 다른 섹션 헤더 발견(III.) at offset 2192
  ⚠ 000660_2022.json: [IV] 다른 섹션 헤더 발견(III.) at offset 2424
  ⚠ 000660_2023.json: [IV] 다른 섹션 헤더 발견(III.) at offset 2064
  ⚠ 000660_2024.json: [IV] 다른 섹션 헤더 발견(III.) at offset 2040
  ⚠ 000860_2022.json: [VI] 다른 섹션 헤더 발견(VIII.) at offset 277
  ⚠ 000860_2023.json: [VI] 다른 섹션 헤더 발견(VIII.) at offset 277
  ⚠ 000860_2

## Cell 16 — 실패 행 재처리 (선택)

`_failed_rows.csv`에 기록된 실패 행을 재시도하려면:

In [40]:
def retry_failures():
    if not FAIL_LOG_PATH.exists():
        print('실패 로그가 없습니다.')
        return
    with open(FAIL_LOG_PATH, 'r', encoding='utf-8') as f:
        failed = list(csv.DictReader(f))
    if not failed:
        print('재시도할 항목이 없습니다.')
        return
    # 백업 후 비우기 (재시도 결과만 새로 기록되도록)
    FAIL_LOG_PATH.rename(FAIL_LOG_PATH.with_suffix('.csv.bak'))

    # 원본 CSV에서 해당 행을 다시 가져와 재처리
    with open(CSV_PATH, 'r', encoding='utf-8-sig') as f:
        all_rows = list(csv.DictReader(f))

    print(f'재시도 대상: {len(failed)}건')
    for row in failed:
        idx = int(row['idx'])
        # 기존 결과 파일이 있으면 삭제 (재생성하기 위해)
        sc = str(row['stock_code']).zfill(6)
        fy = str(row['fiscal_year']).strip()
        out_path = OUTPUT_DIR / f'{sc}_{fy}.json'
        if out_path.exists():
            out_path.unlink()
        status = process_row(idx, all_rows[idx - 1])
        print(f'  idx={idx} {sc} {fy} : {status}')


# retry_failures()  # 필요 시 주석 해제